# 💰 SpendSmart - Phase 5 Model Comparison & Evaluation

This notebook trains and compares 4 distinct machine learning models on SpendSmart's transaction dataset (`combined_training_data.csv`) using an identical 80/20 train/test split:
1. **Multinomial Naive Bayes**
2. **Logistic Regression**
3. **Linear SVM (`LinearSVC`)**
4. **k-Nearest Neighbors (`KNeighborsClassifier`)**

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

## 1. Load Data & Prepare Features

In [2]:
df = pd.read_csv('combined_training_data.csv')
df['title'] = df['title'].fillna('')
df['description'] = df['description'].fillna('')
df['full_text'] = (df['title'] + ' ' + df['description']).str.strip()

X = df['full_text'].values
y = df['category'].values
unique_labels = sorted(list(set(y)))

print(f"Dataset size: {len(X)} samples across categories: {unique_labels}")

## 2. Train-Test Split (80/20 Stratified)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

## 3. Train & Evaluate 4 ML Classifiers

In [4]:
models = {
    'Multinomial Naive Bayes': MultinomialNB(alpha=0.1),
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    'Linear SVM': LinearSVC(C=1.0, random_state=42),
    'k-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5)
}

results = []
confusion_matrices = {}

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes = axes.flatten()

for idx, (name, clf) in enumerate(models.items()):
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=2500, lowercase=True, strip_accents='unicode')),
        ('clf', clf)
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred) * 100
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)
    
    cm = confusion_matrix(y_test, y_pred, labels=unique_labels)
    confusion_matrices[name] = cm
    
    results.append({
        'Model': name,
        'Accuracy (%)': round(acc, 2),
        'Macro Precision (%)': round(prec * 100, 2),
        'Macro Recall (%)': round(rec * 100, 2),
        'Macro F1-Score (%)': round(f1 * 100, 2)
    })
    
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=unique_labels, yticklabels=unique_labels,
        ax=axes[idx], cbar=False
    )
    axes[idx].set_title(f"{name}\nAccuracy: {acc:.2f}% | F1: {f1*100:.2f}%", fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Predicted Category')
    axes[idx].set_ylabel('Actual Category')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=300)
plt.show()

## 4. Final Comparison Results Table

In [5]:
df_results = pd.DataFrame(results)
df_results.to_csv('model_comparison_results.csv', index=False)
df_results